# Multi-output model quick test (mo_tbe)

Notebook to load a multi-output emulator, run predictions on test data, and visualize output-wise errors.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

from NN_emulator import emulator

In [ ]:
BASE_DIR = '/gpfs0/elyk/users/hovavl/GITs/TBE'
MODEL_DIR = f'{BASE_DIR}/models/mo_tbe'
DATA_PATH = f'{MODEL_DIR}/training_files.pk'
MODEL_NAME = 'mo_tbe'

In [ ]:
with open(DATA_PATH, 'rb') as f:
    training_params, features, val_params, val_features, testing_params, testing_features, model_params, output_axis = pickle.load(f)

model = emulator(restore=True, use_log=False, files_dir=MODEL_DIR, name=MODEL_NAME)
test_loss, pred = model.test_APE(testing_params, testing_features)

print('test samples:', testing_features.shape[0])
print('outputs per sample:', testing_features.shape[1])
print('median APE (%):', np.median(test_loss))

In [ ]:
output_axis = np.asarray(output_axis)
sort_idx = np.argsort(output_axis)
sorted_axis = output_axis[sort_idx]
sorted_true = testing_features[:, sort_idx]
sorted_pred = pred[:, sort_idx]

output_ape = 100 * np.abs((sorted_true - sorted_pred) / sorted_true)
median_output_ape = np.median(output_ape, axis=0)

plt.figure(figsize=(10, 4))
plt.plot(sorted_axis, median_output_ape, lw=2)
plt.xlabel('Output axis (sorted)')
plt.ylabel('Median APE [%]')
plt.title('Per-output median error')
plt.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
sample_idx = int(np.argmax(test_loss))

plt.figure(figsize=(10, 4))
plt.plot(sorted_axis, sorted_true[sample_idx], label='True', lw=2)
plt.plot(sorted_axis, sorted_pred[sample_idx], '--', label='Prediction', lw=2)
plt.xlabel('Output axis (sorted)')
plt.ylabel('Output value')
plt.title(f'Worst test sample (APE={test_loss[sample_idx]:.2f}%)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()